# Triadic Cell Notebook v73
## Prompt-Recovered Slot Corpus Auditor

v72 found v71 patched contracts, but it still exposed two corpus-quality Ω states:

$$
\Omega_{prompt}: \text{prompt cavity degraded into gold-choice fallback}
$$

$$
\Omega_{hurt}: \text{hurt-residue file exists but is empty / unreadable}
$$

v73 fixes the exporter by recovering original prompts from embedded notebook data and by separating normal SFT rows from true hurt-repair rows.

Outputs:

```text
slot_training_pairs.jsonl
slot_training_pairs.csv
slot_sft_messages.jsonl
slot_repair_messages.jsonl
slot_preference_pairs.jsonl
slot_hurt_repair_messages.jsonl
slot_corpus_audit.csv
slot_corpus_manifest.json
input_read_status.csv
```

Training target remains:

$$
Q \rightarrow C
$$

not:

$$
Q \rightarrow \text{answer}
$$


In [ ]:
from __future__ import annotations
import json, re
from pathlib import Path
from typing import Optional, List
import numpy as np
import pandas as pd

OUTPUT_DIR = "v73_outputs_prompt_recovered_slot_corpus"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

DOWNLOADS = Path.home() / ""
RUN_ROOTS = [Path.cwd(), DOWNLOADS, DOWNLOADS/"Nexus", DOWNLOADS/"ChatGPT", Path("/mnt/data")]

CANDIDATE_RUN_DIRS = [
    "v71_outputs_qwen25_1p5b_instruct_rotations_hurt_residue_backpatch_controller",
    "v70_outputs_qwen25_1p5b_instruct_rotations_dual_channel_recursive_controller",
    "v69_outputs_qwen25_1p5b_instruct_rotations_recursive_operational_checklist_controller",
    "v68_outputs_qwen25_1p5b_instruct_rotations_recursive_operational_checklist_critic",
    "v67_outputs_qwen25_1p5b_instruct_rotations_clean_recursive_slot_constructor",
    "v64_outputs_qwen25_1p5b_instruct_rotations_self_generating_slot_constructor",
]
NOTEBOOK_PATTERNS = ["triadic_cell_notebook_v*.ipynb", "*triadic*cell*.ipynb", "*nexus*.ipynb"]

def unique_paths(paths):
    out=[]; seen=set()
    for p in paths:
        try: k=str(p.resolve())
        except Exception: k=str(p)
        if k not in seen:
            seen.add(k); out.append(p)
    return out

def find_run_dirs():
    found=[]
    for root in RUN_ROOTS:
        if not root.exists():
            continue
        for name in CANDIDATE_RUN_DIRS:
            p=root/name
            if p.exists() and p.is_dir(): found.append(p)
        for p in root.glob("v*_outputs_*"):
            if p.is_dir(): found.append(p)
    return unique_paths(found)

def find_notebooks():
    found=[]
    for root in RUN_ROOTS:
        if not root.exists(): continue
        for pat in NOTEBOOK_PATTERNS:
            found.extend(root.glob(pat))
    return unique_paths([p for p in found if p.is_file() and p.suffix==".ipynb"])

run_dirs=find_run_dirs()
print("Found run dirs:")
for p in run_dirs: print(" -", p)


In [ ]:
# -----------------------------
# Robust readers
# -----------------------------
EMPTY_HURT_COLUMNS = [
    "row_id","base_id","band","false_idx","gold_idx",
    "false_choice","gold_choice","false_source","gold_source",
    "base_margin","generated_margin","generated_support","op_quality",
    "F_need","F_function","F_boundary","F_trap","F_collapse",
]
read_status={}

def read_csv_if_exists(path: Path, expected_empty_columns: Optional[List[str]]=None):
    if not path.exists():
        read_status[str(path)]="missing"; return None
    try:
        if path.stat().st_size == 0:
            read_status[str(path)]="empty_file"
            return pd.DataFrame(columns=expected_empty_columns or [])
    except Exception:
        pass
    try:
        df=pd.read_csv(path)
        read_status[str(path)]="ok"
        return df
    except pd.errors.EmptyDataError:
        read_status[str(path)]="empty_no_columns"
        return pd.DataFrame(columns=expected_empty_columns or [])
    except Exception as e:
        read_status[str(path)]=f"error:{e}"
        print("Could not read", path, e)
        return None

def write_jsonl(path: Path, rows: List[dict]):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False)+"\n")

def split_pipe_field(x):
    if x is None: return []
    try:
        if pd.isna(x): return []
    except Exception:
        pass
    if isinstance(x, list): return x
    return [p.strip() for p in str(x).split("|") if p.strip()]

def normalize_contract_from_row(row: dict) -> dict:
    residue=row.get("residue", None)
    if residue is None or str(residue).lower() in ["nan","none","null",""]:
        residue=None
    return {
        "family_class": str(row.get("family_class","") or "").strip(),
        "domain_carrier": split_pipe_field(row.get("domain_carrier", "")),
        "forbidden_neighbor_carrier": split_pipe_field(row.get("forbidden_neighbor_carrier", "")),
        "boundary_conditions": split_pipe_field(row.get("boundary_conditions", "")),
        "preserved_function": str(row.get("preserved_function", "") or "").strip(),
        "failure_modes": split_pipe_field(row.get("failure_modes", "")),
        "witness_readout": str(row.get("witness_readout", "") or "").strip(),
        "residue": residue,
    }

def compact_contract(c: dict) -> dict:
    keys=["family_class","domain_carrier","forbidden_neighbor_carrier","boundary_conditions","preserved_function","failure_modes","witness_readout","residue"]
    return {k:c.get(k,None) for k in keys}


In [ ]:
# -----------------------------
# Recover embedded ADVERSARIAL_NEXUS_DATA from local notebooks
# -----------------------------
import nbformat

def nb_version(p):
    m=re.search(r"v(\d+)", p.name)
    return int(m.group(1)) if m else -1

def load_embedded_dataset(nb_path: Path):
    try:
        nb=nbformat.read(nb_path.open("r", encoding="utf-8"), as_version=4)
    except Exception as e:
        return None, None, f"read_error:{e}"
    for cell in nb.cells:
        if cell.cell_type != "code": continue
        src=cell.source
        if "ADVERSARIAL_NEXUS_DATA" in src and "ABSTRACT_SLOT_SPECS_RAW" in src:
            ns={}
            try:
                exec(src, ns, ns)
                data=ns.get("ADVERSARIAL_NEXUS_DATA")
                slots=ns.get("ABSTRACT_SLOT_SPECS_RAW")
                if isinstance(data, list) and len(data):
                    return data, slots, "ok"
            except Exception as e:
                return None, None, f"exec_error:{e}"
    return None, None, "not_found"

notebooks=find_notebooks()
embedded_data=None; embedded_slots=None; embedded_source=None; status=[]
for nbp in sorted(notebooks, key=nb_version, reverse=True):
    data, slots, st=load_embedded_dataset(nbp)
    status.append({"notebook":str(nbp), "status":st, "n":0 if data is None else len(data)})
    if data is not None:
        embedded_data=data; embedded_slots=slots; embedded_source=nbp; break

embedded_status_df=pd.DataFrame(status)
display(embedded_status_df.head(40))
print("embedded_source:", embedded_source)
print("embedded rows:", 0 if embedded_data is None else len(embedded_data))

embedded_prompt_map={}; embedded_gold_map={}; embedded_row_map={}
if embedded_data:
    for i,row in enumerate(embedded_data):
        bid=str(row.get("id", f"row_{i}"))
        embedded_row_map[bid]=row
        embedded_prompt_map[bid]=str(row.get("prompt", ""))
        try: embedded_gold_map[bid]=str(row["choices"][int(row["answer_idx"])])
        except Exception: embedded_gold_map[bid]=""
print("embedded_prompt_map:", len(embedded_prompt_map))


In [ ]:
# -----------------------------
# Load lineage artifacts
# -----------------------------
artifacts=[]
for run_dir in run_dirs:
    patch_dir=run_dir/"v71_hurt_residue_patches"
    artifacts.append({
        "run_dir":run_dir,
        "summary":read_csv_if_exists(run_dir/"summary.csv"),
        "results":read_csv_if_exists(run_dir/"results.csv"),
        "interesting":read_csv_if_exists(run_dir/"interesting_cases.csv"),
        "initial_contracts":read_csv_if_exists(run_dir/"initial_contracts.csv"),
        "repaired_contracts":read_csv_if_exists(run_dir/"repaired_contracts.csv"),
        "operational_repaired_contracts":read_csv_if_exists(run_dir/"operational_repaired_contracts.csv"),
        "operational_audit":read_csv_if_exists(run_dir/"operational_repaired_audit.csv"),
        "operational_trace":read_csv_if_exists(run_dir/"operational_repair_trace.csv"),
        "v71_patch_audit":read_csv_if_exists(run_dir/"v71_patch_audit.csv"),
        "v71_best_eval":read_csv_if_exists(run_dir/"v71_best_eval_rows.csv"),
        "v71_best_fail":read_csv_if_exists(run_dir/"v71_best_fail_rows.csv"),
        "v71_hurt_records":read_csv_if_exists(patch_dir/"hurt_records_from_best_explore.csv", expected_empty_columns=EMPTY_HURT_COLUMNS),
        "v71_patched_contracts":read_csv_if_exists(patch_dir/"patched_contracts.csv"),
    })

manifest_rows=[]
for a in artifacts:
    for k,v in a.items():
        if k=="run_dir": continue
        manifest_rows.append({"run_dir":str(a["run_dir"]),"artifact":k,"found":v is not None,"rows":0 if v is None else len(v)})
manifest_df=pd.DataFrame(manifest_rows)
display(manifest_df[manifest_df.found].sort_values(["run_dir","artifact"]))
read_status_df=pd.DataFrame([{"path":k,"status":v} for k,v in read_status.items()])
display(read_status_df[read_status_df.status.astype(str).str.contains("empty|error", na=False)])


In [ ]:
# -----------------------------
# Select best sources
# -----------------------------
def version_num(p):
    m=re.search(r"v(\d+)_outputs", str(p))
    return int(m.group(1)) if m else -1

def first_nonempty(key, allow_empty=False):
    for a in sorted(artifacts, key=lambda a: version_num(a["run_dir"]), reverse=True):
        df=a.get(key)
        if df is not None and (allow_empty or len(df)):
            return a["run_dir"], df
    return None, None

source_results_dir, results_df = first_nonempty("results")
source_initial_dir, initial_contracts_df = first_nonempty("initial_contracts")
source_operational_dir, operational_contracts_df = first_nonempty("operational_repaired_contracts")
source_repaired_dir, repaired_contracts_df = first_nonempty("repaired_contracts")
source_patched_dir, patched_contracts_df = first_nonempty("v71_patched_contracts")
source_hurt_dir, hurt_records_df = first_nonempty("v71_hurt_records", allow_empty=True)
source_patch_audit_dir, patch_audit_df = first_nonempty("v71_patch_audit")
source_best_eval_dir, best_eval_df = first_nonempty("v71_best_eval")
source_best_fail_dir, best_fail_df = first_nonempty("v71_best_fail")

print("Selected sources:")
for name, src, df in [
    ("results", source_results_dir, results_df),
    ("initial", source_initial_dir, initial_contracts_df),
    ("operational", source_operational_dir, operational_contracts_df),
    ("repaired", source_repaired_dir, repaired_contracts_df),
    ("patched", source_patched_dir, patched_contracts_df),
    ("hurt", source_hurt_dir, hurt_records_df),
    ("patch_audit", source_patch_audit_dir, patch_audit_df),
    ("best_eval", source_best_eval_dir, best_eval_df),
    ("best_fail", source_best_fail_dir, best_fail_df),
]:
    print(name+":", src, None if df is None else len(df))

if results_df is None:
    raise RuntimeError("No results.csv found. Run v71/v70/v68/v67 first, then rerun v73.")


In [ ]:
# -----------------------------
# Maps
# -----------------------------
def df_to_contract_map(df, id_col="base_id"):
    out={}
    if df is None: return out
    for _,r in df.iterrows():
        if id_col not in r or pd.isna(r[id_col]): continue
        out[str(r[id_col])]=normalize_contract_from_row(dict(r))
    return out

initial_map=df_to_contract_map(initial_contracts_df)
operational_map=df_to_contract_map(operational_contracts_df)
repaired_map=df_to_contract_map(repaired_contracts_df)
patched_map=df_to_contract_map(patched_contracts_df)

target_contract_map={}
all_ids=set(embedded_prompt_map.keys()) | set(initial_map) | set(operational_map) | set(repaired_map) | set(patched_map)
for bid in sorted(all_ids):
    if bid in patched_map: target_contract_map[bid]=patched_map[bid]
    elif bid in operational_map: target_contract_map[bid]=operational_map[bid]
    elif bid in repaired_map: target_contract_map[bid]=repaired_map[bid]
    elif bid in initial_map: target_contract_map[bid]=initial_map[bid]

print("maps:", {"initial":len(initial_map),"operational":len(operational_map),"repaired":len(repaired_map),"patched":len(patched_map),"target":len(target_contract_map)})

def is_degraded_prompt(s):
    s=str(s)
    return "Gold choice observed:" in s or "Generate the missing-shape contract for this Nexus task" in s or len(s.strip())<20

prompt_map={}; prompt_source_map={}
for bid,p in embedded_prompt_map.items():
    if p and not is_degraded_prompt(p):
        prompt_map[bid]=p; prompt_source_map[bid]="embedded_adversarial_data"

if results_df is not None and "base_id" in results_df.columns and "prompt" in results_df.columns:
    for _,r in results_df.iterrows():
        bid=str(r.get("base_id", "")); p=str(r.get("prompt", ""))
        if bid and bid not in prompt_map and not is_degraded_prompt(p):
            prompt_map[bid]=p; prompt_source_map[bid]="results_prompt"

for bid in target_contract_map:
    if bid not in prompt_map:
        gold=embedded_gold_map.get(bid, "")
        prompt_map[bid]=f"[DEGRADED_PROMPT_MISSING] base_id={bid}; gold_choice={gold}"
        prompt_source_map[bid]="degraded_fallback"

result_map={}
if results_df is not None and "base_id" in results_df.columns:
    cols=["base_correct","compiled_correct","generated_correct","generated_helped","generated_hurt","generated_omega"]
    existing=[c for c in cols if c in results_df.columns]
    if existing:
        g=results_df.groupby("base_id")[existing].mean().reset_index()
        for _,r in g.iterrows(): result_map[str(r["base_id"])]=dict(r)

hurt_map={}; hurt_file_status="missing"
if hurt_records_df is not None:
    hurt_file_status="empty" if len(hurt_records_df)==0 else "ok"
    if len(hurt_records_df):
        for bid,grp in hurt_records_df.groupby("base_id"):
            hurt_map[str(bid)]=grp.to_dict(orient="records")

audit_map={}
if patch_audit_df is not None and len(patch_audit_df) and "base_id" in patch_audit_df.columns:
    for _,r in patch_audit_df.iterrows():
        audit_map[str(r["base_id"])]={
            "quality":float(r.get("quality",np.nan)),
            "F_need":float(r.get("F_need",np.nan)),
            "F_function":float(r.get("F_function",np.nan)),
            "F_boundary":float(r.get("F_boundary",np.nan)),
            "F_trap":float(r.get("F_trap",np.nan)),
            "F_collapse":float(r.get("F_collapse",np.nan)),
            "failures":split_pipe_field(r.get("failures","")),
        }
print("prompt real:", sum(1 for b in prompt_map if prompt_source_map.get(b)!="degraded_fallback"), "degraded:", sum(1 for b in prompt_map if prompt_source_map.get(b)=="degraded_fallback"))
print("hurt_map:", len(hurt_map), "hurt_file_status:", hurt_file_status)


In [ ]:
# -----------------------------
# Corpus audit
# -----------------------------
def contract_complete(c):
    if not c: return False
    for k in ["family_class","domain_carrier","forbidden_neighbor_carrier","boundary_conditions","preserved_function","failure_modes","witness_readout"]:
        v=c.get(k)
        if isinstance(v, list):
            if not [x for x in v if str(x).strip()]: return False
        elif not str(v or "").strip():
            return False
    return True

def gold_leak(prompt):
    return "Gold choice observed:" in str(prompt) or "[DEGRADED_PROMPT_MISSING]" in str(prompt)

audit_rows=[]
for bid,c in sorted(target_contract_map.items()):
    prompt=prompt_map.get(bid, "")
    audit_rows.append({
        "base_id":bid,
        "prompt_real":prompt_source_map.get(bid)!="degraded_fallback",
        "prompt_source":prompt_source_map.get(bid,"unknown"),
        "gold_choice_leak":gold_leak(prompt),
        "patched_contract_present":bid in patched_map,
        "hurt_residue_present":bid in hurt_map and len(hurt_map[bid])>0,
        "contract_complete":contract_complete(c),
        "target_source":"patched" if bid in patched_map else "operational_repaired" if bid in operational_map else "repaired" if bid in repaired_map else "initial",
    })
corpus_audit_df=pd.DataFrame(audit_rows)
display(corpus_audit_df)

corpus_ready_sft=bool(len(corpus_audit_df)>0 and corpus_audit_df.prompt_real.all() and (~corpus_audit_df.gold_choice_leak).all() and corpus_audit_df.contract_complete.all())
corpus_ready_hurt=bool(corpus_ready_sft and corpus_audit_df.hurt_residue_present.any())
print("SFT corpus ready:", corpus_ready_sft)
print("Hurt-repair corpus ready:", corpus_ready_hurt)


In [ ]:
# -----------------------------
# Training message templates
# -----------------------------
SYSTEM_SLOT_BUILDER = """You are the Nexus Slot Constructor.

Your job is to generate the missing-shape contract before answer selection.
Do not answer the task.
Do not mention answer choices.
Return strict JSON only.

The contract must contain:
family_class
domain_carrier
forbidden_neighbor_carrier
boundary_conditions
preserved_function
failure_modes
witness_readout
residue

Use operational fit, not labels.
"""

def make_user_prompt_for_sft(prompt):
    return f"""Prompt:
{prompt}

Generate the missing-shape contract.

Checklist:
1. Need: occupy the inverse cavity.
2. Function: preserve or redirect the required operation.
3. Boundary: respect constraints.
4. Trap: reject noun/surface-label confusion.
5. Collapse: produce one executable witness/readout.

Return JSON only."""

def make_user_prompt_for_repair(prompt, initial_contract, audit, hurt_residue):
    return f"""Prompt:
{prompt}

Current contract:
{json.dumps(compact_contract(initial_contract), ensure_ascii=False, indent=2)}

Operational audit:
{json.dumps(audit, ensure_ascii=False, indent=2)}

Hurt / Ω residue:
{json.dumps(hurt_residue, ensure_ascii=False, indent=2)}

Repair the contract only.
Do not answer the task.
Return strict JSON only."""

def to_messages(system, user, assistant_obj):
    return {"messages":[
        {"role":"system","content":system},
        {"role":"user","content":user},
        {"role":"assistant","content":json.dumps(compact_contract(assistant_obj), ensure_ascii=False, indent=2)},
    ]}


In [ ]:
# -----------------------------
# Build and save corpus
# -----------------------------
slot_training_pairs=[]; slot_sft_messages=[]; slot_repair_messages=[]; slot_preference_pairs=[]; slot_hurt_repair_messages=[]
for bid,target in sorted(target_contract_map.items()):
    prompt=prompt_map.get(bid, "")
    initial=initial_map.get(bid,{})
    operational=operational_map.get(bid,{})
    repaired=repaired_map.get(bid,{})
    patched=patched_map.get(bid,{})
    audit=audit_map.get(bid,{})
    hurt_residue=hurt_map.get(bid, [])
    result=result_map.get(bid,{})
    target_source="patched" if bid in patched_map else "operational_repaired" if bid in operational_map else "repaired" if bid in repaired_map else "initial"
    ca=corpus_audit_df[corpus_audit_df.base_id==bid].iloc[0].to_dict()
    row={
        "base_id":bid,"prompt":prompt,"prompt_source":prompt_source_map.get(bid,"unknown"),
        "initial_contract":compact_contract(initial) if initial else None,
        "operational_contract":compact_contract(operational) if operational else None,
        "repaired_contract":compact_contract(repaired) if repaired else None,
        "patched_contract":compact_contract(patched) if patched else None,
        "target_contract":compact_contract(target),"target_source":target_source,
        "operational_audit":audit,"hurt_residue":hurt_residue,"result":result,"corpus_audit":ca,
    }
    slot_training_pairs.append(row)
    if ca["prompt_real"] and not ca["gold_choice_leak"] and ca["contract_complete"]:
        msg=to_messages(SYSTEM_SLOT_BUILDER, make_user_prompt_for_sft(prompt), target)
        msg.update({"base_id":bid,"target_source":target_source,"prompt_source":prompt_source_map.get(bid,"unknown")})
        slot_sft_messages.append(msg)
    if initial and target and ca["prompt_real"] and not ca["gold_choice_leak"]:
        repair=to_messages(SYSTEM_SLOT_BUILDER, make_user_prompt_for_repair(prompt, initial, audit, hurt_residue), target)
        repair.update({"base_id":bid,"target_source":target_source,"prompt_source":prompt_source_map.get(bid,"unknown")})
        slot_repair_messages.append(repair)
        if compact_contract(initial)!=compact_contract(target):
            slot_preference_pairs.append({
                "base_id":bid,
                "prompt":make_user_prompt_for_repair(prompt, initial, audit, hurt_residue),
                "chosen":json.dumps(compact_contract(target), ensure_ascii=False, indent=2),
                "rejected":json.dumps(compact_contract(initial), ensure_ascii=False, indent=2),
                "metadata":{"target_source":target_source,"has_hurt_residue":bool(hurt_residue),"audit":audit,"result":result,"prompt_source":prompt_source_map.get(bid,"unknown")},
            })
        if hurt_residue:
            hmsg=to_messages(SYSTEM_SLOT_BUILDER, make_user_prompt_for_repair(prompt, initial, audit, hurt_residue), target)
            hmsg.update({"base_id":bid,"target_source":target_source,"prompt_source":prompt_source_map.get(bid,"unknown")})
            slot_hurt_repair_messages.append(hmsg)

out=Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
write_jsonl(out/"slot_training_pairs.jsonl", slot_training_pairs)
write_jsonl(out/"slot_sft_messages.jsonl", slot_sft_messages)
write_jsonl(out/"slot_repair_messages.jsonl", slot_repair_messages)
write_jsonl(out/"slot_preference_pairs.jsonl", slot_preference_pairs)
write_jsonl(out/"slot_hurt_repair_messages.jsonl", slot_hurt_repair_messages)

csv_rows=[]
for r in slot_training_pairs:
    tc=r["target_contract"]; audit=r.get("operational_audit",{}) or {}; result=r.get("result",{}) or {}; ca=r.get("corpus_audit",{}) or {}
    csv_rows.append({
        "base_id":r["base_id"],"target_source":r["target_source"],"prompt_source":r["prompt_source"],"prompt":r["prompt"],
        "prompt_real":ca.get("prompt_real",False),"gold_choice_leak":ca.get("gold_choice_leak",True),"patched_contract_present":ca.get("patched_contract_present",False),"hurt_residue_present":ca.get("hurt_residue_present",False),"contract_complete":ca.get("contract_complete",False),
        "family_class":tc.get("family_class",""),"domain_carrier":" | ".join(tc.get("domain_carrier",[]) or []),"forbidden_neighbor_carrier":" | ".join(tc.get("forbidden_neighbor_carrier",[]) or []),"boundary_conditions":" | ".join(tc.get("boundary_conditions",[]) or []),"preserved_function":tc.get("preserved_function",""),"failure_modes":" | ".join(tc.get("failure_modes",[]) or []),"witness_readout":tc.get("witness_readout",""),"residue":tc.get("residue",None),
        "quality":audit.get("quality",np.nan),"F_need":audit.get("F_need",np.nan),"F_function":audit.get("F_function",np.nan),"F_boundary":audit.get("F_boundary",np.nan),"F_trap":audit.get("F_trap",np.nan),"F_collapse":audit.get("F_collapse",np.nan),"generated_correct":result.get("generated_correct",np.nan),"generated_hurt":result.get("generated_hurt",np.nan),"generated_helped":result.get("generated_helped",np.nan),
    })
corpus_df=pd.DataFrame(csv_rows)
corpus_df.to_csv(out/"slot_training_pairs.csv", index=False)
corpus_audit_df.to_csv(out/"slot_corpus_audit.csv", index=False)
pd.DataFrame([{"path":k,"status":v} for k,v in read_status.items()]).to_csv(out/"input_read_status.csv", index=False)

manifest={
    "output_dir":str(out),
    "n_slot_training_pairs":len(slot_training_pairs),
    "n_slot_sft_messages":len(slot_sft_messages),
    "n_slot_repair_messages":len(slot_repair_messages),
    "n_slot_preference_pairs":len(slot_preference_pairs),
    "n_slot_hurt_repair_messages":len(slot_hurt_repair_messages),
    "sft_corpus_ready":corpus_ready_sft,
    "hurt_repair_corpus_ready":corpus_ready_hurt,
    "embedded_prompt_source":str(embedded_source) if embedded_source else None,
    "source_results_dir":str(source_results_dir) if source_results_dir else None,
    "source_initial_dir":str(source_initial_dir) if source_initial_dir else None,
    "source_operational_dir":str(source_operational_dir) if source_operational_dir else None,
    "source_patched_dir":str(source_patched_dir) if source_patched_dir else None,
    "source_hurt_dir":str(source_hurt_dir) if source_hurt_dir else None,
    "hurt_file_status":hurt_file_status,
    "quality_counts":{
        "prompt_real_rows":int(corpus_audit_df.prompt_real.sum()),
        "gold_leak_rows":int(corpus_audit_df.gold_choice_leak.sum()),
        "patched_rows":int(corpus_audit_df.patched_contract_present.sum()),
        "hurt_rows":int(corpus_audit_df.hurt_residue_present.sum()),
        "complete_rows":int(corpus_audit_df.contract_complete.sum()),
    },
    "files":{
        "slot_training_pairs_jsonl":"slot_training_pairs.jsonl",
        "slot_training_pairs_csv":"slot_training_pairs.csv",
        "slot_sft_messages_jsonl":"slot_sft_messages.jsonl",
        "slot_repair_messages_jsonl":"slot_repair_messages.jsonl",
        "slot_preference_pairs_jsonl":"slot_preference_pairs.jsonl",
        "slot_hurt_repair_messages_jsonl":"slot_hurt_repair_messages.jsonl",
        "slot_corpus_audit_csv":"slot_corpus_audit.csv",
        "input_read_status_csv":"input_read_status.csv",
    }
}
(out/"slot_corpus_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(corpus_df.head(24))
print(json.dumps(manifest, indent=2))


## Readout

If the manifest says:

```json
"sft_corpus_ready": true
```

then train on:

```text
slot_sft_messages.jsonl
```

If it says:

```json
"hurt_repair_corpus_ready": true
```

then train repair on:

```text
slot_hurt_repair_messages.jsonl
```

If SFT is ready but hurt repair is false, the state is:

$$
Q_{real}\rightarrow C_{patched}
$$

available, but:

$$
(Q,C_0,A_0,\Omega_{hurt})\rightarrow C_1
$$

not yet available.
